# 04 — Optimization & Miscellaneous

贝叶斯优化收敛历史 + 电压-响应度曲线等杂项。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.interpolate import CubicSpline
from pathlib import Path

In [ ]:
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'stix'

## 1. 电压 — 响应度曲线

In [ ]:
volt = np.array([35, 40, 45, 47])
eff  = np.array([1.24, 1.57, 2.74, 1.09])

order = np.argsort(volt)
volt_s = volt[order]
eff_s = eff[order]

cs = CubicSpline(volt_s, eff_s)
x_fit = np.linspace(volt_s.min(), volt_s.max(), 400)
y_fit = cs(x_fit)

plt.figure(figsize=(7, 5))
plt.scatter(volt, eff, color='red', s=80, zorder=5)
plt.plot(x_fit, y_fit, linestyle='--', color='black', linewidth=2, zorder=3)
plt.xlabel('Voltage (V)', fontsize=16)
plt.ylabel('Responsivity (A/W)', fontsize=16)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2. 贝叶斯优化收敛历史

读取 BO 日志 CSV，绘制 f 值随迭代的收敛。

In [ ]:
csv_path = Path(r"D:\xwechat_files\wxid_umtzlz2f1fad22_9bf9\msg\file\2025-11\bend_optimization_log.csv")
df = pd.read_csv(csv_path)

# 容错：取最后一列作为 f
if "f" in df.columns:
    f = df["f"].astype(float).values
else:
    f = df.iloc[:, -1].astype(float).values

it = np.arange(1, len(f) + 1)
best_so_far = np.minimum.accumulate(f)
best_idx = int(np.argmin(f))
best_val = float(f[best_idx])

plt.figure(figsize=(8, 5))
plt.plot(it, f, '.-', alpha=0.6, label='f (raw)')
plt.plot(it, best_so_far, '--', linewidth=1.5, label='best-so-far (cummin)')
plt.scatter([best_idx + 1], [best_val], color='red', zorder=10)
plt.annotate(f"best={best_val:.4f}\nit={best_idx + 1}",
             (best_idx + 1, best_val),
             xytext=(best_idx + 5, best_val + 0.02),
             arrowprops=dict(arrowstyle="->"), fontsize=9)
plt.xlabel("Evaluation index")
plt.ylabel("f")
plt.title("Optimization f history")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()